# SpaceAmb — Prototype Demo

Architectural-semantic research prototype.  Full pipeline:

1. Load atoms, programs, ambiances, scenes
2. Generate embeddings (disk-cached)
3. Score atoms, descriptors, and scenes against all space × ambiance queries
4. Explore: change settings in **Section 0b** to point at any query or scene
5. Evaluate: Hit@1 / Hit@5 ground-truth metrics for scene retrieval
6. Export and visualise

> **Methodological note:** Similarity scores reflect *semantic affinity* in
> embedding space — a research instrument, not perceptual ground truth.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve().parent
src_path = ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

## 0b. Settings  <-- **edit here to explore different queries & scenes**

Change `EXPLORE_SPACE` and `EXPLORE_AMBIANCE` to drill into any combination.
`EXAMPLE_QUERIES` drives the comparison charts and exports throughout the notebook.

Valid values are any `text` entries in `data/raw/programs.json` / `data/raw/ambiances.json`.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PRIMARY EXPLORATION TARGET
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Space (program) — must match a text value in data/raw/programs.json
EXPLORE_SPACE    = 'living room'
# Ambiance — must match a text value in data/raw/ambiances.json
EXPLORE_AMBIANCE = 'relaxing'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMPARISON QUERIES (used in charts, exports, heatmap)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE_QUERIES = [
    'relaxing living room',
    'somber conference room',
    'vibrant gym',
    'spooky gallery',
]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ATOM TO TRACK ACROSS ALL QUERIES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CROSS_QUERY_ATOM = 'sofa'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SCENE TO DEEP-DIVE  (must match an id in data/raw/scenes.json)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXPLORE_SCENE_ID = 'scene_relaxing_living_room_01'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DISPLAY DEPTH
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TOP_K        = 20   # atoms / descriptors per query
TOP_K_SCENES = 5    # scenes per query

# Derived — do not edit
EXPLORE_QUERY = f'{EXPLORE_AMBIANCE} {EXPLORE_SPACE}'
print(f'Explore query  : {EXPLORE_QUERY!r}')
print(f'Example queries: {EXAMPLE_QUERIES}')

## 1. Load Config and Datasets

In [ ]:
import yaml
from semantic_architecture.atoms import load_atoms, atoms_summary
from semantic_architecture.queries import load_programs, load_ambiances
from semantic_architecture.scenes import load_scenes, scenes_summary, validate_scenes

CONFIG_PATH = ROOT / 'config' / 'config.yaml'
with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

atoms     = load_atoms(ROOT / cfg['data']['atoms_path'])
programs  = load_programs(ROOT / cfg['data']['programs_path'])
ambiances = load_ambiances(ROOT / cfg['data']['ambiances_path'])
scenes    = load_scenes(ROOT / cfg['data']['scenes_path'])

print(atoms_summary(atoms))
print(f"\nPrograms  ({len(programs)}): {[p['text'] for p in programs]}")
print(f"Ambiances ({len(ambiances)}): {[a['text'] for a in ambiances]}")
print()
print(scenes_summary(scenes))

# Validate scenes
valid_spaces = [p['text'] for p in programs]
valid_ambs   = [a['text'] for a in ambiances]
for w in validate_scenes(scenes, valid_spaces, valid_ambs):
    print(f'WARNING: {w}')

## 2. Generate Embeddings (Cached)

In [ ]:
from semantic_architecture.embeddings import EmbeddingModel
from semantic_architecture.queries import generate_all_queries

emb_cfg = cfg['embedding']
model = EmbeddingModel(
    model_name=emb_cfg['model_name'],
    cache_dir=ROOT / emb_cfg['cache_dir'],
    batch_size=emb_cfg['batch_size'],
)

query_sets   = generate_all_queries(programs, ambiances)
space_qs     = query_sets['space']
ambiance_qs  = query_sets['ambiance']
combined_qs  = query_sets['combined']

atom_texts    = [a.text for a in atoms]
atom_families = [a.family for a in atoms]
atom_ids      = [a.id for a in atoms]

# Use embedding_text (disambiguation-aware) for all query embeddings
atom_embs     = model.load_or_compute(atom_texts,                                   'atoms')
space_embs    = model.load_or_compute([q.embedding_text for q in space_qs],         'queries_space')
ambiance_embs = model.load_or_compute([q.embedding_text for q in ambiance_qs],      'queries_ambiance')
combined_embs = model.load_or_compute([q.embedding_text for q in combined_qs],      'queries_combined')

scene_texts    = [s.text for s in scenes]
scene_families = [s.space for s in scenes]
scene_ids      = [s.id for s in scenes]
scene_embs     = model.load_or_compute(scene_texts, 'scenes')

print(f'Atoms      : {atom_embs.shape}')
print(f'Queries    : {len(space_qs)} space, {len(ambiance_qs)} ambiance, {len(combined_qs)} combined')
print(f'Scenes     : {scene_embs.shape}')

In [ ]:
# Build query-text -> query-id lookup (used throughout the notebook)
q_lookup = {q.combined_text: q.id for q in combined_qs}

# Resolve EXAMPLE_QUERIES -> IDs
ex_ids = {}
for qt in EXAMPLE_QUERIES:
    if qt not in q_lookup:
        print(f"WARNING: '{qt}' not found - check programs/ambiances data")
    else:
        ex_ids[qt] = q_lookup[qt]

# Resolve EXPLORE_QUERY -> ID
if EXPLORE_QUERY not in q_lookup:
    print(f"WARNING: EXPLORE_QUERY '{EXPLORE_QUERY}' not found - check EXPLORE_SPACE / EXPLORE_AMBIANCE settings")
    explore_qid = None
else:
    explore_qid = q_lookup[EXPLORE_QUERY]

print(f'Explore query ID : {explore_qid}')
print(f'Example query IDs: {list(ex_ids.keys())}')

## 3. Score Atoms

In [ ]:
from semantic_architecture.scoring import (
    ScoringWeights, score_items_against_queries, enrich_with_discriminative_scores
)
from semantic_architecture.analysis import (
    top_atoms_for_query, top_atoms_by_family, compare_item_across_queries
)

weights = ScoringWeights.from_config(cfg.get('scoring', {}))

atom_scores = score_items_against_queries(
    item_texts=atom_texts, item_families=atom_families, item_ids=atom_ids,
    item_embeddings=atom_embs,
    space_queries=space_qs, ambiance_queries=ambiance_qs, combined_queries=combined_qs,
    space_embeddings=space_embs, ambiance_embeddings=ambiance_embs,
    combined_embeddings=combined_embs,
    weights=weights,
)
atom_scores = enrich_with_discriminative_scores(atom_scores)
print(f'Atom score table: {atom_scores.shape}')
print('Columns:', list(atom_scores.columns))

In [ ]:
# Top atoms for EXPLORE_QUERY (discriminative ranking)
# Change EXPLORE_SPACE / EXPLORE_AMBIANCE in the Settings cell to target a different query.
top_atoms = top_atoms_for_query(explore_qid, atom_scores, k=TOP_K,
                                 score_col='discriminative_score')
print(f'Top {TOP_K} atoms for {EXPLORE_QUERY!r} (discriminative):')
top_atoms

In [ ]:
# Top atoms grouped by family for EXPLORE_QUERY
by_fam = top_atoms_by_family(explore_qid, atom_scores, k=5,
                               score_col='discriminative_score')
for fam, df in sorted(by_fam.items()):
    print(f'\n--- {fam} ---')
    # Only show columns that are available in the returned dataframe
    show_cols = [c for c in ['text', 'discriminative_score'] if c in df.columns]
    print(df[show_cols].to_string(index=False))

In [ ]:
# Cross-query: how does CROSS_QUERY_ATOM score across all 400 queries?
# Change CROSS_QUERY_ATOM in the Settings cell to track a different atom.
atom_cross = compare_item_across_queries(CROSS_QUERY_ATOM, atom_scores)
print(f"Top-10 queries for '{CROSS_QUERY_ATOM}':")
print(atom_cross.head(10)[['combined_text', 'sim_space', 'sim_ambiance',
                             'sim_combined', 'weighted_score', 'discriminative_score']
                          ].to_string(index=False))
print(f"\nBottom-5 queries for '{CROSS_QUERY_ATOM}':")
print(atom_cross.tail(5)[['combined_text', 'discriminative_score']].to_string(index=False))

## 4. Generate Descriptors

In [ ]:
from semantic_architecture.composition import generate_descriptors, descriptors_to_df

comp_cfg = cfg.get('composition', {})
descriptors = generate_descriptors(
    atoms=atoms,
    atom_embeddings=atom_embs,
    n_descriptors=comp_cfg.get('n_descriptors', 300),
    descriptor_lengths=comp_cfg.get('descriptor_lengths', [2, 3]),
    temperature=comp_cfg.get('temperature', 1.0),
    seed=comp_cfg.get('seed', 42),
)

desc_df = descriptors_to_df(descriptors)
print(f'Generated {len(descriptors)} unique descriptors')
print(f'\nFamily patterns (top 15):')
print(desc_df['family_pattern'].value_counts().head(15).to_string())
desc_df[['text', 'family_pattern', 'n_atoms']].head(20)

In [ ]:
# Show full provenance for the first 5 descriptors
for d in descriptors[:5]:
    print(d.provenance_str())
    print()

## 5. Score Descriptors

In [ ]:
from semantic_architecture.analysis import top_descriptors_for_query

desc_texts    = [d.text for d in descriptors]
desc_embs     = model.load_or_compute(desc_texts, 'descriptors')
desc_families = [d.source_atom_families[0] for d in descriptors]
desc_ids      = [d.id for d in descriptors]

descriptor_scores = score_items_against_queries(
    item_texts=desc_texts, item_families=desc_families, item_ids=desc_ids,
    item_embeddings=desc_embs,
    space_queries=space_qs, ambiance_queries=ambiance_qs, combined_queries=combined_qs,
    space_embeddings=space_embs, ambiance_embeddings=ambiance_embs,
    combined_embeddings=combined_embs,
    weights=weights,
)
descriptor_scores = enrich_with_discriminative_scores(descriptor_scores)
print(f'Descriptor score table: {descriptor_scores.shape}')

In [ ]:
# Top descriptors for EXPLORE_QUERY
top_d = top_descriptors_for_query(explore_qid, descriptor_scores, k=TOP_K,
                                   score_col='discriminative_score')
print(f'Top {TOP_K} descriptors for {EXPLORE_QUERY!r}:')
cols = ['rank', 'text', 'family', 'weighted_score', 'discriminative_score']
top_d[[c for c in cols if c in top_d.columns]]

In [ ]:
# Descriptors for all EXAMPLE_QUERIES side-by-side
for q_text in EXAMPLE_QUERIES:
    if q_text not in ex_ids:
        print(f"Skip: '{q_text}' not in ex_ids")
        continue
    top_d = top_descriptors_for_query(ex_ids[q_text], descriptor_scores, k=10,
                                       score_col='discriminative_score')
    print(f'\n-- Top 10 descriptors for {q_text!r} --')
    cols = ['rank', 'text', 'family', 'discriminative_score']
    print(top_d[[c for c in cols if c in top_d.columns]].to_string(index=False))

## 6. Score Scenes

Each scene has an **intended query** (its `space` x `ambiance` pair).
Scoring scenes against all 400 combined queries lets us check whether
the embedding space can retrieve the right context for a given description.

In [ ]:
from semantic_architecture.analysis import top_scenes_for_query

scene_scores = score_items_against_queries(
    item_texts=scene_texts, item_families=scene_families, item_ids=scene_ids,
    item_embeddings=scene_embs,
    space_queries=space_qs, ambiance_queries=ambiance_qs, combined_queries=combined_qs,
    space_embeddings=space_embs, ambiance_embeddings=ambiance_embs,
    combined_embeddings=combined_embs,
    weights=weights,
)
scene_scores = enrich_with_discriminative_scores(scene_scores)
print(f'Scene score table: {scene_scores.shape}')

## 7. Scene Explorer

Use **`EXPLORE_SPACE` / `EXPLORE_AMBIANCE`** and **`EXPLORE_SCENE_ID`**
from the Settings cell to focus on any space x ambiance pair or scene.

In [ ]:
# -- 7a. Scene browser: all scenes with their intended vs predicted query --
rows = []
for s in scenes:
    row = scene_scores[scene_scores['item_id'] == s.id].copy()
    row = row.sort_values('discriminative_score', ascending=False)
    top_query = row.iloc[0]['combined_text'] if len(row) > 0 else '-'
    rows.append({
        'id': s.id,
        'intended': s.intended_query,
        'top_predicted': top_query,
        'hit@1': top_query == s.intended_query,
        'preview': s.text[:90] + '...',
    })
browser_df = pd.DataFrame(rows)
print(f'{len(scenes)} scenes loaded')
browser_df

In [ ]:
# -- 7b. Top scenes for EXPLORE_QUERY --
# Change EXPLORE_SPACE / EXPLORE_AMBIANCE in the Settings cell.
if explore_qid:
    top_s = top_scenes_for_query(explore_qid, scene_scores, k=TOP_K_SCENES,
                                  score_col='discriminative_score')
    print(f'Top {TOP_K_SCENES} scenes for {EXPLORE_QUERY!r}:')
    show_cols = [c for c in ['rank', 'text', 'family', 'discriminative_score'] if c in top_s.columns]
    print(top_s[show_cols].to_string(index=False))
else:
    print('EXPLORE_QUERY not resolved - check settings.')

In [ ]:
# -- 7c. Top scenes for each EXAMPLE_QUERY --
for q_text in EXAMPLE_QUERIES:
    if q_text not in ex_ids:
        continue
    top_s = top_scenes_for_query(ex_ids[q_text], scene_scores, k=TOP_K_SCENES,
                                  score_col='discriminative_score')
    print(f'\n-- Top {TOP_K_SCENES} scenes for {q_text!r} --')
    show_cols = [c for c in ['rank', 'text', 'family', 'discriminative_score'] if c in top_s.columns]
    print(top_s[show_cols].to_string(index=False))

In [ ]:
# -- 7d. Scene deep-dive: full text + top queries for EXPLORE_SCENE_ID --
# Change EXPLORE_SCENE_ID in the Settings cell to inspect a different scene.
scene_match = [s for s in scenes if s.id == EXPLORE_SCENE_ID]
if not scene_match:
    print(f"Scene '{EXPLORE_SCENE_ID}' not found - check EXPLORE_SCENE_ID in Settings")
else:
    scene = scene_match[0]
    print(f'Scene  : {scene.id}')
    print(f'Intended query: {scene.intended_query!r}')
    if scene.notes:
        print(f'Notes  : {scene.notes}')
    print()
    print('-- Full text --')
    print(scene.text)
    print()
    s_row = scene_scores[scene_scores['item_id'] == scene.id].copy()
    s_row = s_row.sort_values('discriminative_score', ascending=False).head(10)
    print('-- Top 10 matching queries (discriminative) --')
    print(s_row[['combined_text', 'discriminative_score', 'weighted_score']].to_string(index=False))

In [ ]:
# -- 7e. All scenes ranked for EXPLORE_QUERY (full ranking table) --
if explore_qid:
    all_scene_rows = scene_scores[scene_scores['query_id'] == explore_qid].copy()
    all_scene_rows = all_scene_rows.sort_values('discriminative_score', ascending=False)
    display_cols = [c for c in ['item_id', 'text', 'sim_space', 'sim_ambiance',
                                 'sim_combined', 'weighted_score',
                                 'discriminative_score'] if c in all_scene_rows.columns]
    all_scene_rows = all_scene_rows[display_cols].reset_index(drop=True)
    all_scene_rows.index += 1
    print(f'All {len(scenes)} scenes ranked for {EXPLORE_QUERY!r}:')
    all_scene_rows

## 8. Ground-Truth Evaluation

For each scene, does the pipeline rank its intended `space x ambiance` query first?

In [ ]:
results = []
for scene in scenes:
    intended = scene.intended_query
    s_row = scene_scores[scene_scores['item_id'] == scene.id].copy()
    s_row = s_row.sort_values('discriminative_score', ascending=False).reset_index(drop=True)

    top1_text = s_row.iloc[0]['combined_text']
    intended_mask = s_row['combined_text'] == intended
    rank_of_intended = int(s_row.index[intended_mask][0]) + 1 if intended_mask.any() else None

    results.append({
        'scene_id': scene.id,
        'intended': intended,
        'top1_query': top1_text,
        'hit@1': top1_text == intended,
        'rank_of_intended': rank_of_intended,
    })

eval_df = pd.DataFrame(results)
hit1 = eval_df['hit@1'].mean()
hit5 = (eval_df['rank_of_intended'] <= 5).mean()
print(f'Hit@1  : {hit1:.0%}  ({eval_df["hit@1"].sum()}/{len(eval_df)})')
print(f'Hit@5  : {hit5:.0%}  ({(eval_df["rank_of_intended"] <= 5).sum()}/{len(eval_df)})')
print()
eval_df[['scene_id', 'intended', 'top1_query', 'hit@1', 'rank_of_intended']]

## 9. Export All Results

In [ ]:
from semantic_architecture.io_utils import save_csv, save_json

out_dir = ROOT / cfg['output']['dir']
out_dir.mkdir(parents=True, exist_ok=True)

save_csv(atom_scores,       out_dir / 'atom_scores.csv',       'atom scores')
save_csv(descriptor_scores, out_dir / 'descriptor_scores.csv', 'descriptor scores')
save_csv(desc_df,           out_dir / 'descriptors.csv',       'descriptor table')
save_csv(scene_scores,      out_dir / 'scene_scores.csv',      'scene scores')
save_csv(eval_df,           out_dir / 'scene_eval.csv',        'scene evaluation')

# Per-query rankings for EXAMPLE_QUERIES
for q_text, qid in ex_ids.items():
    safe = q_text.replace(' ', '_')
    save_csv(top_atoms_for_query(qid, atom_scores, k=30),
             out_dir / f'ranking_atoms_{safe}.csv')
    save_csv(top_descriptors_for_query(qid, descriptor_scores, k=30),
             out_dir / f'ranking_descriptors_{safe}.csv')
    save_csv(top_scenes_for_query(qid, scene_scores, k=len(scenes)),
             out_dir / f'ranking_scenes_{safe}.csv')

print('Export complete.')

## 10. Visualizations

In [ ]:
from semantic_architecture.visualization import (
    plot_top_items, plot_heatmap, plot_pca_projection,
    plot_umap_projection, plot_family_comparison,
)
from semantic_architecture.scoring import similarity_matrix_df

In [ ]:
# -- 10a. Bar chart: top atoms for EXPLORE_QUERY --
fig = plot_top_items(
    top_atoms_for_query(explore_qid, atom_scores, k=TOP_K,
                        score_col='discriminative_score'),
    query_label=EXPLORE_QUERY, top_k=TOP_K,
    score_col='discriminative_score', color_by_family=True,
)
safe_q = EXPLORE_QUERY.replace(' ', '_')
fig.savefig(out_dir / f'bar_atoms_{safe_q}.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# -- 10b. Heatmap: top-25 atoms x EXAMPLE_QUERIES --
ex_combined_embs, ex_labels = [], []
for qt in EXAMPLE_QUERIES:
    idx = next((i for i, q in enumerate(combined_qs) if q.combined_text == qt), None)
    if idx is not None:
        ex_combined_embs.append(combined_embs[idx])
        ex_labels.append(qt)
ex_combined_embs = np.stack(ex_combined_embs)

ex_qids = list(ex_ids.values())
mean_scores = (
    atom_scores[atom_scores['query_id'].isin(ex_qids)]
    .groupby('text')['weighted_score'].mean()
    .sort_values(ascending=False)
    .head(25)
)
top25_texts = list(mean_scores.index)
top25_idx   = [atom_texts.index(t) for t in top25_texts]
top25_embs  = atom_embs[top25_idx]

mat = similarity_matrix_df(top25_texts, ex_labels, top25_embs, ex_combined_embs)
fig = plot_heatmap(mat, title=f'Top-25 atoms vs {len(ex_labels)} example queries',
                   figsize=(10, 10), annot=True)
fig.savefig(out_dir / 'heatmap_atoms_example_queries.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# -- 10c. PCA of all atoms --
fig = plot_pca_projection(
    atom_embs, atom_texts, families=atom_families,
    title='PCA of all atoms (coloured by family)',
    annotate=True, figsize=(12, 9),
)
fig.savefig(out_dir / 'pca_atoms.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# -- 10d. UMAP (skipped if umap-learn not installed) --
fig = plot_umap_projection(
    atom_embs, atom_texts, families=atom_families,
    title='UMAP of all atoms (coloured by family)',
    annotate=True, figsize=(12, 9),
)
if fig:
    fig.savefig(out_dir / 'umap_atoms.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# -- 10e. Family comparison across EXAMPLE_QUERIES --
fig = plot_family_comparison(
    atom_scores, query_ids=ex_qids,
    score_col='weighted_score', figsize=(14, 6),
)
fig.savefig(out_dir / 'family_comparison_example_queries.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# -- 10f. PCA atoms + scenes overlaid --
all_embs     = np.vstack([atom_embs, scene_embs])
all_texts    = atom_texts + [f'[{s.id}]' for s in scenes]
all_families = atom_families + [f'scene:{s.space}' for s in scenes]

fig = plot_pca_projection(
    all_embs, all_texts, families=all_families,
    title='PCA - atoms (dots) + scenes (triangles) overlaid',
    annotate=False, figsize=(13, 10),
)
fig.savefig(out_dir / 'pca_atoms_scenes.png', bbox_inches='tight', dpi=150)
plt.show()

## Summary

Outputs saved to `data/processed/`:

| File | Contents |
|------|----------|
| `atom_scores.csv` | Atoms x 400 queries |
| `descriptor_scores.csv` | Descriptors x 400 queries |
| `scene_scores.csv` | Scenes x 400 queries |
| `scene_eval.csv` | Hit@1 / Hit@5 ground-truth evaluation |
| `descriptors.csv` | Descriptor list with provenance |
| `ranking_atoms_*.csv` | Per-query atom rankings |
| `ranking_descriptors_*.csv` | Per-query descriptor rankings |
| `ranking_scenes_*.csv` | Per-query scene rankings |
| `bar_atoms_*.png` | Bar chart for explore query |
| `heatmap_atoms_example_queries.png` | Heatmap |
| `pca_atoms.png` | PCA projection (atoms only) |
| `pca_atoms_scenes.png` | PCA projection (atoms + scenes) |
| `umap_atoms.png` | UMAP projection |
| `family_comparison_example_queries.png` | Family comparison chart |

> **To explore a different query:** change `EXPLORE_SPACE` and `EXPLORE_AMBIANCE`
> in the Settings cell (section 0b), then re-run sections 3-7.
>
> **To explore a different scene:** change `EXPLORE_SCENE_ID` in the Settings cell.
>
> **To add comparison queries:** add entries to `EXAMPLE_QUERIES` in the Settings cell.